# 00 · Download, Verify & Inventory
### *From Diagnosis to Decision* — dataset acquisition for ICA 2026

This notebook downloads the datasets in **priority order**, runs a verification
block for each, and appends a row to `data/manifest.csv`. It follows the
acquisition spec in `DATASETS.md`.

**Run order:** run this notebook first. The EDA / analysis notebooks
(`01`–`05`) adapt to whatever datasets are present, so you can start analysis
after only the Tier-1 (or the minimal-viable) subset has downloaded.

**Environment:** designed for Google Colab (free T4, Google Drive for
persistence). `data/raw/` is **git-ignored** — never commit image data; only
`manifest.csv` and the label mappings are versioned.

> ⚠️ Downloads are large and credential-gated. Read §1 (credentials) before running.

## 1 · Configuration

In [ ]:
# --- Environment config: works on Google Colab AND locally --------------------
import os, sys, pathlib

def in_colab():
    return "google.colab" in sys.modules or os.path.exists("/content")

if in_colab():
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = pathlib.Path("/content/drive/MyDrive/diagnosis-to-decision")
else:
    # local fallback: repo root (edit if you cloned elsewhere)
    PROJECT_ROOT = pathlib.Path(
        os.environ.get("ICA_PROJECT_ROOT", pathlib.Path.cwd().parents[0])
    )

DATA_RAW     = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_MAPPING = PROJECT_ROOT / "data" / "mapping"
FIGDIR       = PROJECT_ROOT / "reports" / "figures"
for p in (DATA_RAW, DATA_INTERIM, DATA_MAPPING, FIGDIR):
    p.mkdir(parents=True, exist_ok=True)

print("Colab:", in_colab())
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_RAW exists:", DATA_RAW.exists())

In [ ]:
# Run once per Colab session. Locally, install into a venv instead.
%pip install -q datasets huggingface_hub kaggle roboflow imagehash pillow tqdm pandas scikit-learn scikit-image matplotlib

## 2 · Credentials

Set these up once. Nothing here is committed.

| Service | Needed for | How |
|---|---|---|
| **Kaggle** | Cassava, Master dataset | Kaggle → Settings → *Create New API Token* → upload `kaggle.json` below |
| **Hugging Face** | PlantVillage, PlantWild | `huggingface-cli login` (paste a read token) |
| **Roboflow** | FieldPlant | Free account → Settings → API key |

Zenodo, GitHub and Mendeley need **no** authentication.

In [ ]:
# --- Kaggle: upload kaggle.json (Colab) or place it at ~/.kaggle/kaggle.json
import os, pathlib
KAGGLE_DIR = pathlib.Path.home() / ".kaggle"
KAGGLE_DIR.mkdir(exist_ok=True)
if in_colab() and not (KAGGLE_DIR / "kaggle.json").exists():
    from google.colab import files
    print("Upload your kaggle.json:")
    up = files.upload()
    if "kaggle.json" in up:
        (KAGGLE_DIR / "kaggle.json").write_bytes(up["kaggle.json"])
        os.chmod(KAGGLE_DIR / "kaggle.json", 0o600)
print("kaggle.json present:", (KAGGLE_DIR / "kaggle.json").exists())

In [ ]:
# --- Hugging Face login (needed for gated repos; PlantVillage/PlantWild are public
#     but a token avoids rate limits). Skip if you prefer anonymous access.
try:
    from huggingface_hub import login, whoami
    HF_TOKEN = os.environ.get("HF_TOKEN", "")   # or paste here / use Colab secret
    if HF_TOKEN:
        login(HF_TOKEN)
        print("HF user:", whoami()["name"])
    else:
        print("No HF token set — proceeding anonymously (fine for public sets).")
except Exception as e:
    print("HF login skipped:", e)

## 3 · Manifest helper

In [ ]:
import csv, hashlib, datetime as dt
MANIFEST = PROJECT_ROOT / "data" / "manifest.csv"
FIELDS = ["name","tier","source","local_path","n_files","n_classes",
          "size_gb","license","sha256_of_archive","download_utc","status"]

def _dir_stats(path):
    exts = {".jpg",".jpeg",".png",".bmp",".tif",".tiff"}
    n_files, size = 0, 0
    subdirs = set()
    for p in pathlib.Path(path).rglob("*"):
        if p.is_file():
            size += p.stat().st_size
            if p.suffix.lower() in exts:
                n_files += 1
                subdirs.add(p.parent.name)
    return n_files, len(subdirs), round(size/1e9, 3)

def record(name, tier, source, path, license, status="ok",
           n_classes=None, sha256=None, when_utc=None):
    n_files, guessed_classes, size_gb = _dir_stats(path)
    row = {
        "name": name, "tier": tier, "source": source, "local_path": str(path),
        "n_files": n_files,
        "n_classes": n_classes if n_classes is not None else guessed_classes,
        "size_gb": size_gb, "license": license,
        "sha256_of_archive": sha256 or "",
        # NOTE: pass a real timestamp string; avoid nondeterministic calls in CI
        "download_utc": when_utc or "SET_AT_RUNTIME",
        "status": status,
    }
    new = not MANIFEST.exists()
    with open(MANIFEST, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        if new: w.writeheader()
        w.writerow(row)
    print("manifest +=", row)
    return row

## 4 · TIER 1 — required for the paper

Download these four first. Nothing in the study runs without them.

### 4.1 PlantVillage  *(Hugging Face — the leak-safe source)*

⚠️ **Data-leakage warning.** PlantVillage has multiple photos of the *same
physical leaf*. A random split puts one leaf on both sides and inflates accuracy.
Use the **`leaf_id`** field to build leaf-grouped splits (see notebook `04`).
Do **not** use the Kaggle mirrors — they drop `leaf_id`.

In [ ]:
from datasets import load_dataset
PV_DIR = DATA_RAW / "plantvillage"; PV_DIR.mkdir(parents=True, exist_ok=True)

# "color" is the standard config; "grayscale" and "segmented" also exist.
pv = load_dataset("mohanty/PlantVillage", "color")
print(pv)

# --- Verify ---
n_total = sum(len(pv[s]) for s in pv)
print("total images:", n_total)
assert n_total >= 54000, "expected >=54,000 images"
feats = pv[list(pv)[0]].features
print("has leaf_id:", "leaf_id" in feats)          # MUST be True for leak-safe splits
labels = set()
for s in pv: labels |= set(pv[s]["label"])
print("n classes:", len(labels))                    # expect 38

# Cache to disk for the other notebooks
pv.save_to_disk(str(PV_DIR / "hf_arrow"))
record("PlantVillage", 1, "hf:mohanty/PlantVillage", PV_DIR,
       "CC BY-SA 3.0", n_classes=len(labels))

### 4.2 PlantWild  *(Hugging Face — primary in-the-wild eval)*

License **CC BY-NC-ND 4.0**: metrics only — do **not** redistribute repackaged
images. The strongest published method reaches only ~52% here, so it is the main
generalization target and a rich source of hard cases for the abstention gate.

In [ ]:
from datasets import load_dataset
PW_DIR = DATA_RAW / "plantwild"; PW_DIR.mkdir(parents=True, exist_ok=True)
pw = load_dataset("uqtwei2/PlantWild")
print(pw)
n = sum(len(pw[s]) for s in pw)
print("total images:", n)
assert 18000 <= n <= 19000, f"unexpected count {n}"
pw.save_to_disk(str(PW_DIR / "hf_arrow"))
record("PlantWild", 1, "hf:uqtwei2/PlantWild", PW_DIR, "CC BY-NC-ND 4.0")

### 4.3 PlantSeg  *(Zenodo — pixel masks → severity)*

⚠️ **Confirm the Zenodo record before bulk download.** Multiple records
circulate. Open the landing page, confirm which one is the *complete* dataset
(images + annotations + `Metadata.csv`), and record the DOI you used.
Do not use the Roboflow copy (truncated).

In [ ]:
# PlantSeg is ~6 GB. Prefer Kaggle CLI (resumable) or Zenodo direct.
PS_DIR = DATA_RAW / "plantseg"; PS_DIR.mkdir(parents=True, exist_ok=True)

# Option A — Kaggle (recommended, resumable):
#   !kaggle datasets download -d weitianqi/plantseg -p {PS_DIR} --unzip
# Option B — Zenodo direct (CONFIRM the record id on the landing page first):
#   ZENODO_RECORD = "13762907"   # <-- verify this is the complete dataset
#   !wget -c "https://zenodo.org/records/{ZENODO_RECORD}/files/..." -P {PS_DIR}

# --- Verify (after download) ---
import glob
imgs = list((PS_DIR).rglob("*.jpg")) + list((PS_DIR).rglob("*.png"))
meta = list(PS_DIR.rglob("Metadata.csv"))
print("image-ish files:", len(imgs), "| Metadata.csv found:", bool(meta))
if imgs and meta:
    record("PlantSeg", 1, "zenodo:CONFIRM_RECORD", PS_DIR, "CC BY-NC 4.0")
else:
    print("Not downloaded yet — uncomment a download option above.")

### 4.4 PlantDoc  *(GitHub — second field test set, cleanest license)*

CC BY 4.0, ~300 MB, no auth. Some images are actually lab shots and annotation
was done without a plant pathologist → expect label noise (a threat to validity
to disclose in the paper).

In [ ]:
PD_DIR = DATA_RAW / "plantdoc"
if not PD_DIR.exists() or not any(PD_DIR.rglob("*.jpg")):
    !git clone --depth 1 https://github.com/pratikkayal/PlantDoc-Dataset.git "{PD_DIR}"
# --- Verify ---
n_imgs = len(list(PD_DIR.rglob("*.jpg")) + list(PD_DIR.rglob("*.JPG")) +
             list(PD_DIR.rglob("*.png")))
n_train_cls = len([d for d in (PD_DIR/'train').iterdir() if d.is_dir()]) if (PD_DIR/'train').exists() else 0
print("images:", n_imgs, "| train class dirs:", n_train_cls)
record("PlantDoc", 1, "github:pratikkayal/PlantDoc-Dataset", PD_DIR,
       "CC BY 4.0", n_classes=n_train_cls)

## 5 · Minimal-viable additions (Tier 2–3)

If disk/time is tight, these three complete every experimental component:
**Cassava** (viral classes for the harm matrix), **RoCoLe** or **BRACOL**
(severity validation, both < 2 GB).

In [ ]:
# --- Cassava (Kaggle competition; accept the rules on the website first) ---
CAS_DIR = DATA_RAW / "cassava"; CAS_DIR.mkdir(parents=True, exist_ok=True)
# !kaggle competitions download -c cassava-leaf-disease-classification -p {CAS_DIR}
# !cd {CAS_DIR} && unzip -q -o cassava-leaf-disease-classification.zip
if any(CAS_DIR.rglob("*.jpg")):
    record("Cassava", 2, "kaggle:cassava-leaf-disease-classification",
           CAS_DIR, "competition rules", n_classes=5)
else:
    print("Cassava not downloaded — accept comp rules, then uncomment above.")

In [ ]:
# --- RoCoLe (Mendeley, no auth): severity labels + leaf masks -> validates the
#     mask->severity pipeline end to end. DOI 10.17632/c5yvn32dzg.2
RC_DIR = DATA_RAW / "rocole"; RC_DIR.mkdir(parents=True, exist_ok=True)
# Mendeley gives a signed zip URL from the landing page "Download All" button;
# paste it here (they expire), or download manually and unzip into RC_DIR.
# !wget -c "PASTE_MENDELEY_ZIP_URL" -O {RC_DIR}/rocole.zip && unzip -q -o {RC_DIR}/rocole.zip -d {RC_DIR}
print("RoCoLe files:", len(list(RC_DIR.rglob("*"))))

## 6 · Inventory

`manifest.csv` is the single source of truth for what was actually downloaded.
Commit **only** this file (and the label mappings) — never the images.

In [ ]:
import pandas as pd
if MANIFEST.exists():
    display(pd.read_csv(MANIFEST))
else:
    print("No manifest yet — run the download cells above.")

---
**Next:** `01_eda_classification_sets.ipynb` (EDA), then `02`–`05` for severity,
leakage/dedup, and the baseline "pre-training" results.